In [ ]:
# ==========================================================
# STEP 1: Setup & Import Libraries
# ==========================================================
import requests
import pandas as pd
from datetime import datetime

# ==========================================================
# STEP 2: Configuration (Edit these for your project!)
# ==========================================================
# Options for SEARCH_TYPE: 'team', 'user', or 'location'
SEARCH_TYPE = 'team' 

# Project Details
TEAM_NAME = "Udea0311152"  # Case sensitive
USER_ID = "12345678"              # Your GLOBE User ID
START_DATE = "2026-01-01"
END_DATE = "2026-02-26"

# Location (Calle 47 Medellín area)
LAT, LON = 6.244, -75.573
DISTANCE_KM = 5  # Search radius

# ==========================================================
# STEP 3: API Query Logic
# ==========================================================
base_url = "https://api.globe.gov/search/v1/measurement/protocol/measureddate/"

# Determine endpoint based on SEARCH_TYPE
if SEARCH_TYPE == 'team':
    url = f"{base_url}globeteams/?teams={TEAM_NAME}"
elif SEARCH_TYPE == 'user':
    url = f"{base_url}userid/?userid={USER_ID}"
elif SEARCH_TYPE == 'location':
    url = f"{base_url}point/distance/?lat={LAT}&lon={LON}&distancekm={DISTANCE_KM}"

# Standard parameters for Cloud Data
params = {
    "protocols": "sky_conditions",
    "startdate": START_DATE,
    "enddate": END_DATE,
    "geojson": "TRUE",
    "sample": "FALSE"
}

print(f"📡 Querying GLOBE API for {SEARCH_TYPE} data...")

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    features = data.get('features', [])
    
    # ==========================================================
    # STEP 4: Data Processing & Display
    # ==========================================================
    if not features:
        print("❌ No observations found for this criteria.")
    else:
        # Convert GeoJSON to a flat list for Pandas
        rows = []
        for f in features:
            properties = f['properties']
            # We add coordinates manually from the geometry part of the GeoJSON
            properties['latitude'] = f['geometry']['coordinates'][1]
            properties['longitude'] = f['geometry']['coordinates'][0]
            rows.append(properties)
        
        df = pd.DataFrame(rows)
        
        print("-" * 30)
        print(f"✅ SUCCESS: Retrieved {len(df)} total observations.")
        print("-" * 30)
        
        # Display basic info: Top 5 rows
        # We select specific columns to keep the display clean
        display_cols = ['userName', 'measuredDate', 'cloudType', 'latitude', 'longitude']
        available_cols = [c for c in display_cols if c in df.columns]
        
        print("\n--- Preview of Data ---")
        display(df[available_cols].head())
        
        # Summary statistics
        if 'userName' in df.columns:
            print("\n--- Submissions per Student ---")
            print(df['userName'].value_counts())

else:
    print(f"🚨 API Error: {response.status_code}")
    print(response.text)